# CheXpert RAD-DINO pathology projection pre-adaptation

This notebook adapts `microsoft/rad-dino-maira-2` to six retrieval
pathologies plus a strict normal cohort. It follows the DenseNet
notebook's CheXpert data protocol: frontal images only, patient-level
train/dev/internal-validation split, explicit uncertain-label policy,
class balancing and development-only checkpoint selection.

The checkpoint is intended as external pre-adaptation. It must not be
evaluated as an IU-Xray final model without an IU-Xray group hold-out.


In [ ]:
# 1. Imports and deterministic setup
import json
import math
import os
import random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import average_precision_score, roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler
from transformers import AutoImageProcessor, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


In [ ]:
# 2. Configuration - change only these paths and limits first
# Kaggle mount used by the ashery/chexpert dataset.
KAGGLE_DATASET_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
DATA_ROOT = (
    KAGGLE_DATASET_ROOT / "CheXpert-v1.0-small"
    if (KAGGLE_DATASET_ROOT / "CheXpert-v1.0-small" / "train.csv").is_file()
    else KAGGLE_DATASET_ROOT
)
IMAGE_ROOT = DATA_ROOT
TRAIN_CSV = DATA_ROOT / "train.csv"
OFFICIAL_VALID_CSV = DATA_ROOT / "valid.csv"
OUT_DIR = Path("/kaggle/working/chexpert_rad_dino_projection")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "microsoft/rad-dino-maira-2"
HF_TOKEN = os.getenv("HF_TOKEN")
MAIN_LABELS = [
    "Atelectasis", "Cardiomegaly", "Edema", "Pleural Effusion",
    "Pneumonia", "Pneumothorax",
]
COHORTS = MAIN_LABELS + ["normal"]

# Same spirit as the DenseNet notebook. Uncertain labels used by the
# classification loss are explicit; metric supervision only uses raw 1.
UNCERTAIN_POLICY = {
    "Atelectasis": "positive",
    "Cardiomegaly": "ignore",
    "Edema": "positive",
    "Pleural Effusion": "ignore",
    "Pneumonia": "ignore",
    "Pneumothorax": "ignore",
}
NAN_TRAIN_POLICY = "ignore"  # "ignore" is safer for external pre-adaptation.
# Normal fallback avoids an empty cohort after frontal filtering and patient split.
NORMAL_POLICY = "strict"  # "strict", "strict_or_target_negative", "target_negative"
TRAIN_RATIO, DEV_RATIO = 0.80, 0.10

BATCH_SIZE = 16       # Must be >= 14 when balanced sampling uses 7 cohorts.
NUM_WORKERS = 4
EPOCHS = 3
# 3 x 800 balanced updates targets about two hours from the measured T4 throughput.
STEPS_PER_EPOCH = 800
LR_HEAD = 1e-4
LR_ENCODER = 1e-5
WEIGHT_DECAY = 1e-5
UNFREEZE_LAST_BLOCKS = 2
POS_WEIGHT_CAP = 10.0

CLASSIFICATION_WEIGHT = 1.0
METRIC_WEIGHT = 1.0
STRUCTURE_WEIGHT = 0.20
NEGATIVE_MARGIN = 0.10
HARD_NEGATIVE_RATIO = 1.0
GRADED_POWER = 2.0
RETRIEVAL_EVAL_MAX_QUERIES = 500
CHECKPOINT_PATH = OUT_DIR / "rad_dino_chexpert_projection_best.pt"
print("Checkpoint:", CHECKPOINT_PATH)


## Label semantics

The seven-dimensional metric support uses only explicit pathology
positives. A normal sample is strict: `No Finding == 1` and all six
target labels are explicit zero. This prevents unknown or uncertain
cases from becoming normal supervision.


In [ ]:
# 3. Load frontal CheXpert rows and create patient-disjoint splits
assert TRAIN_CSV.is_file(), f"Missing {TRAIN_CSV}"
frame = pd.read_csv(TRAIN_CSV)
required = ["Path", "Frontal/Lateral", *MAIN_LABELS]
missing = [name for name in required if name not in frame.columns]
assert not missing, f"Missing columns: {missing}"
HAS_NO_FINDING = "No Finding" in frame.columns

frame = frame[frame["Frontal/Lateral"].astype(str).str.lower().eq("frontal")].copy()
frame["PatientID"] = frame["Path"].str.extract(r"(patient\d+)", expand=False)
assert frame["PatientID"].notna().all(), "Could not extract PatientID from Path"

def numeric(row, name):
    return pd.to_numeric(pd.Series([row[name]]), errors="coerce").iloc[0]

def resolve_value(label, raw):
    if raw == 1:
        return 1.0, 1.0
    if raw == 0:
        return 0.0, 1.0
    if raw == -1:
        policy = UNCERTAIN_POLICY[label]
        if policy == "positive":
            return 1.0, 1.0
        if policy == "negative":
            return 0.0, 1.0
        if policy == "ignore":
            return 0.0, 0.0
        raise ValueError(f"Unsupported uncertain policy for {label}: {policy}")
    if pd.isna(raw):
        if NAN_TRAIN_POLICY == "negative":
            return 0.0, 1.0
        if NAN_TRAIN_POLICY == "ignore":
            return 0.0, 0.0
        raise ValueError(f"Unsupported NAN_TRAIN_POLICY: {NAN_TRAIN_POLICY}")
    raise ValueError(f"Unexpected CheXpert label for {label}: {raw}")

def make_targets(row):
    raw_values = {label: numeric(row, label) for label in MAIN_LABELS}
    has_target_positive = any(value == 1 for value in raw_values.values())
    has_target_uncertain = any(value == -1 for value in raw_values.values())
    target_negative_or_missing = not has_target_positive and not has_target_uncertain
    # CheXpert commonly records non-findings as NaN when No Finding=1.
    strict_normal = bool(
        HAS_NO_FINDING and numeric(row, "No Finding") == 1 and target_negative_or_missing
    )
    if NORMAL_POLICY == "strict":
        normal = strict_normal
    elif NORMAL_POLICY == "target_negative":
        normal = target_negative_or_missing
    elif NORMAL_POLICY == "strict_or_target_negative":
        normal = strict_normal or target_negative_or_missing
    else:
        raise ValueError(f"Unsupported NORMAL_POLICY: {NORMAL_POLICY}")

    targets, masks, metric_support = [], [], []
    for label, raw in raw_values.items():
        target, mask = resolve_value(label, raw)
        targets.append(target)
        masks.append(mask)
        metric_support.append(float(raw == 1))

    targets.append(float(normal))
    masks.append(1.0 if normal or has_target_positive else 0.0)
    metric_support.append(float(normal))
    return targets, masks, metric_support

supervision = frame.apply(make_targets, axis=1)
frame["targets"] = supervision.map(lambda value: value[0])
frame["masks"] = supervision.map(lambda value: value[1])
frame["metric_support"] = supervision.map(lambda value: value[2])
frame = frame[frame["metric_support"].map(sum).gt(0)].reset_index(drop=True)

patients = np.array(sorted(frame["PatientID"].unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(patients)
train_end = int(len(patients) * TRAIN_RATIO)
dev_end = train_end + int(len(patients) * DEV_RATIO)
train_patients = set(patients[:train_end])
dev_patients = set(patients[train_end:dev_end])
valid_patients = set(patients[dev_end:])
train_df = frame[frame.PatientID.isin(train_patients)].reset_index(drop=True)
dev_df = frame[frame.PatientID.isin(dev_patients)].reset_index(drop=True)
valid_df = frame[frame.PatientID.isin(valid_patients)].reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": ["train", "dev", "internal_valid"],
    "images": [len(train_df), len(dev_df), len(valid_df)],
    "patients": [train_df.PatientID.nunique(), dev_df.PatientID.nunique(), valid_df.PatientID.nunique()],
})
print(split_summary.to_string(index=False))
print("Metric cohorts in train:", np.stack(train_df.metric_support.to_numpy()).sum(axis=0).astype(int))


In [ ]:
# 4. Dataset, RAD-DINO processor, and balanced sampler
processor = AutoImageProcessor.from_pretrained(MODEL_NAME, token=HF_TOKEN)

def resolve_image_path(value):
    raw = Path(str(value).replace("\\", "/"))
    parts = raw.parts
    without_dataset_prefix = (
        Path(*parts[1:])
        if parts and parts[0].lower() == "chexpert-v1.0-small"
        else raw
    )
    candidates = [
        raw,
        IMAGE_ROOT / raw,
        IMAGE_ROOT / without_dataset_prefix,
        DATA_ROOT / raw,
        DATA_ROOT / without_dataset_prefix,
        KAGGLE_DATASET_ROOT / raw,
        KAGGLE_DATASET_ROOT / without_dataset_prefix,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    attempted = "\n  - ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(
        f"Cannot resolve CheXpert image from CSV path: {value}\nAttempted:\n  - {attempted}"
    )


class CheXpertProjectionDataset(Dataset):
    def __init__(self, dataframe):
        self.frame = dataframe.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(resolve_image_path(row.Path)) as image:
            image = image.convert("RGB").copy()
        return {
            "image": image,
            "targets": torch.tensor(row.targets, dtype=torch.float32),
            "masks": torch.tensor(row.masks, dtype=torch.float32),
            "support": torch.tensor(row.metric_support, dtype=torch.float32),
            "patient_id": row.PatientID,
        }

def collate(batch):
    encoded = processor(images=[item["image"] for item in batch], return_tensors="pt")
    return {
        "pixels": encoded["pixel_values"],
        "targets": torch.stack([item["targets"] for item in batch]),
        "masks": torch.stack([item["masks"] for item in batch]),
        "support": torch.stack([item["support"] for item in batch]),
        "patient_ids": [item["patient_id"] for item in batch],
    }

class BalancedCohortSampler(Sampler):
    def __init__(self, support, batch_size, seed, steps_per_epoch):
        self.support = np.asarray(support, dtype=np.float32)
        self.batch_size = batch_size
        self.seed = seed
        self.steps_per_epoch = steps_per_epoch
        self.epoch = 0
        if batch_size < 2 * len(COHORTS):
            raise ValueError("BATCH_SIZE must be at least 14 for two samples per cohort")
        if steps_per_epoch < 1:
            raise ValueError("STEPS_PER_EPOCH must be positive")
        self.members = [np.flatnonzero(self.support[:, index] > 0) for index in range(len(COHORTS))]
        empty = [COHORTS[index] for index, indices in enumerate(self.members) if len(indices) == 0]
        if empty:
            counts = {COHORTS[index]: int(len(indices)) for index, indices in enumerate(self.members)}
            raise ValueError(
                f"Empty train cohorts: {empty}. Counts: {counts}. "
                "Review label availability, split sizes, or NORMAL_POLICY."
            )
    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        all_indices = np.arange(len(self.support))
        for _ in range(self.steps_per_epoch):
            selected = []
            for indices in self.members:
                selected.extend(rng.choice(indices, size=2, replace=len(indices) < 2).tolist())
            selected.extend(rng.choice(all_indices, size=self.batch_size - len(selected), replace=False).tolist())
            rng.shuffle(selected)
            yield selected
    def __len__(self):
        return self.steps_per_epoch


In [ ]:
# 5. RAD-DINO model: last blocks + CLS/local patch fusion + 512-D projection
def select_last_blocks(model, count):
    for path in ("encoder.layer", "encoder.layers", "dinov2.encoder.layer", "vision_model.encoder.layer"):
        module = model
        try:
            for name in path.split("."):
                module = getattr(module, name)
        except AttributeError:
            continue
        if isinstance(module, (nn.ModuleList, nn.Sequential)) and len(module) >= count:
            return list(module[-count:])
    raise RuntimeError("Could not locate RAD-DINO transformer blocks")

class ClsPatchFusion(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.LayerNorm(hidden_size * 2), nn.Linear(hidden_size * 2, hidden_size),
            nn.GELU(), nn.Linear(hidden_size, hidden_size),
        )
    def forward(self, cls_token, patch_tokens):
        local_mean = patch_tokens.mean(dim=1)
        return cls_token + self.layers(torch.cat((cls_token, local_mean), dim=1))

class PathologyProjectionHead(nn.Module):
    # State-dict keys match src.embeddings.pathology_projection.PathologyProjectionHead.
    def __init__(self, input_dim, output_dim=512):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(input_dim, output_dim), nn.GELU(), nn.Linear(output_dim, output_dim))
    def forward(self, features):
        return F.normalize(self.layers(features), p=2, dim=1)

class RadDinoProjectionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN)
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False
        self.trainable_blocks = select_last_blocks(self.encoder, UNFREEZE_LAST_BLOCKS)
        for block in self.trainable_blocks:
            for parameter in block.parameters():
                parameter.requires_grad = True
        hidden_size = int(self.encoder.config.hidden_size)
        self.fusion = ClsPatchFusion(hidden_size)
        self.projection = PathologyProjectionHead(hidden_size, 512)
        self.classifier = nn.Linear(512, len(COHORTS))
    def forward(self, pixels):
        hidden = self.encoder(pixel_values=pixels).last_hidden_state
        raw_cls = hidden[:, 0]
        features = self.fusion(raw_cls, hidden[:, 1:])
        embedding = self.projection(F.normalize(features, p=2, dim=1))
        return embedding, self.classifier(embedding), raw_cls

def weighted_jaccard(support):
    numerator = torch.minimum(support[:, None], support[None, :]).sum(dim=-1)
    denominator = torch.maximum(support[:, None], support[None, :]).sum(dim=-1)
    return numerator / denominator.clamp_min(1e-8)

def metric_loss(embedding, support, raw_cls):
    if len(embedding) < 2:
        return embedding.new_zeros(())
    similarity = embedding @ embedding.T
    relevance = weighted_jaccard(support)
    diagonal = torch.eye(len(embedding), dtype=torch.bool, device=embedding.device)
    positive = (relevance > 0) & ~diagonal
    negative = (relevance == 0) & ~diagonal
    zero = embedding.new_zeros(())
    if positive.any():
        weights = relevance[positive].pow(GRADED_POWER)
        positive_loss = (weights * (similarity[positive] - relevance[positive]).square()).sum() / weights.sum().clamp_min(1e-8)
    else:
        positive_loss = zero
    hard_negative = similarity[negative]
    if hard_negative.numel():
        count = min(hard_negative.numel(), max(1, math.ceil(positive.sum().item() * HARD_NEGATIVE_RATIO)))
        hard_negative = torch.topk(hard_negative, count).values
        negative_loss = F.relu(hard_negative - NEGATIVE_MARGIN).square().mean()
    else:
        negative_loss = zero
    raw = F.normalize(raw_cls.detach(), p=2, dim=1)
    structure = F.mse_loss(similarity[~diagonal], (raw @ raw.T)[~diagonal])
    return positive_loss + negative_loss + STRUCTURE_WEIGHT * structure


In [ ]:
# 6. Data loaders, class weights, and optimizer
train_support = np.stack(train_df.metric_support.to_numpy())
train_ds = CheXpertProjectionDataset(train_df)
dev_ds = CheXpertProjectionDataset(dev_df)
valid_ds = CheXpertProjectionDataset(valid_df)
train_loader = DataLoader(train_ds, batch_sampler=BalancedCohortSampler(train_support, BATCH_SIZE, SEED, STEPS_PER_EPOCH), num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", collate_fn=collate)
print(f"Training schedule: {EPOCHS} epochs x {len(train_loader):,} balanced batches = {EPOCHS * len(train_loader):,} updates")
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", collate_fn=collate)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", collate_fn=collate)

target_matrix = np.stack(train_df.targets.to_numpy())
mask_matrix = np.stack(train_df.masks.to_numpy())
positives = (target_matrix * mask_matrix).sum(axis=0)
negatives = ((1 - target_matrix) * mask_matrix).sum(axis=0)
pos_weight = torch.tensor(np.minimum(negatives / np.maximum(positives, 1), POS_WEIGHT_CAP), dtype=torch.float32, device=DEVICE)
print(pd.DataFrame({"cohort": COHORTS, "pos_weight": pos_weight.cpu().numpy()}).round(3))

model = RadDinoProjectionModel().to(DEVICE)
head_params = list(model.fusion.parameters()) + list(model.projection.parameters()) + list(model.classifier.parameters())
encoder_params = [parameter for parameter in model.encoder.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW([
    {"params": head_params, "lr": LR_HEAD},
    {"params": encoder_params, "lr": LR_ENCODER},
], weight_decay=WEIGHT_DECAY)
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))


In [ ]:
# 7. Development metrics and training loop
def safe_auc(y_true, score):
    return float(roc_auc_score(y_true, score)) if len(np.unique(y_true)) == 2 else np.nan

@torch.inference_mode()
def collect(loader):
    model.eval()
    logits, targets, masks, supports, patients, embeddings = [], [], [], [], [], []
    for batch in loader:
        z, output, _ = model(batch["pixels"].to(DEVICE, non_blocking=True))
        logits.append(output.cpu()); targets.append(batch["targets"]); masks.append(batch["masks"])
        supports.append(batch["support"]); patients.extend(batch["patient_ids"]); embeddings.append(z.cpu())
    return {
        "logits": torch.cat(logits).numpy(), "targets": torch.cat(targets).numpy(),
        "masks": torch.cat(masks).numpy(), "support": torch.cat(supports).numpy(),
        "patients": patients, "embeddings": torch.cat(embeddings).numpy(),
    }

def dev_summary(data):
    probabilities = 1 / (1 + np.exp(-data["logits"]))
    auc_rows = []
    for index, label in enumerate(COHORTS):
        valid = data["masks"][:, index] > 0
        y, p = data["targets"][valid, index], probabilities[valid, index]
        auc_rows.append({"label": label, "auroc": safe_auc(y, p), "auprc": float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else np.nan})

    z, support, patients = data["embeddings"], data["support"], data["patients"]
    query_count = min(len(z), RETRIEVAL_EVAL_MAX_QUERIES)
    # Only query_count rows are evaluated; avoid allocating a full N x N similarity matrix.
    similarities = z[:query_count] @ z.T
    ndcgs = []
    for query in range(query_count):
        if support[query].sum() == 0:
            continue
        numerator = np.minimum(support[query], support).sum(axis=1)
        denominator = np.maximum(support[query], support).sum(axis=1)
        relevance = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 0)
        valid = np.asarray([patient != patients[query] for patient in patients])
        candidate_indices = np.flatnonzero(valid)
        ranked = relevance[candidate_indices[np.argsort(-similarities[query, candidate_indices])]]
        top = ranked[:5]
        ideal = np.sort(ranked)[::-1][:5]
        discount = np.log2(np.arange(2, len(top) + 2))
        denominator_dcg = (ideal / discount).sum()
        if denominator_dcg:
            ndcgs.append(float((top / discount).sum() / denominator_dcg))
    table = pd.DataFrame(auc_rows)
    return table, float(np.nanmean(table.auroc)), float(np.nanmean(table.auprc)), float(np.mean(ndcgs)) if ndcgs else np.nan

best_score, best_state, history = -np.inf, None, []
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for batch in train_loader:
        pixels = batch["pixels"].to(DEVICE, non_blocking=True)
        targets = batch["targets"].to(DEVICE, non_blocking=True)
        masks = batch["masks"].to(DEVICE, non_blocking=True)
        support = batch["support"].to(DEVICE, non_blocking=True)
        embedding, logits, raw_cls = model(pixels)
        classification = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction="none")
        classification = (classification * masks).sum() / masks.sum().clamp_min(1)
        metric = metric_loss(embedding, support, raw_cls)
        loss = CLASSIFICATION_WEIGHT * classification + METRIC_WEIGHT * metric
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))

    dev_data = collect(dev_loader)
    dev_table, macro_auroc, macro_auprc, dev_ndcg5 = dev_summary(dev_data)
    row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "dev_macro_auroc": macro_auroc, "dev_macro_auprc": macro_auprc, "dev_nDCG@5": dev_ndcg5}
    history.append(row)
    print(pd.DataFrame([row]).round(5).to_string(index=False))
    print(dev_table.round(4).to_string(index=False))
    if not np.isnan(dev_ndcg5) and dev_ndcg5 > best_score:
        best_score = dev_ndcg5
        best_state = deepcopy(model.state_dict())

assert best_state is not None, "No valid dev nDCG@5 was produced"
model.load_state_dict(best_state)


In [ ]:
# 8. Save a project-compatible pre-adaptation checkpoint and development outputs
block_ids = {id(block) for block in model.trainable_blocks}
block_prefixes = [name for name, module in model.encoder.named_modules() if id(module) in block_ids]
encoder_state = {
    name: value.cpu() for name, value in model.encoder.state_dict().items()
    if any(name == prefix or name.startswith(prefix + ".") for prefix in block_prefixes)
}
torch.save({
    "schema_version": 2,
    "base_model": MODEL_NAME,
    "input_dim": int(model.encoder.config.hidden_size),
    "output_dim": 512,
    "pipeline": "rad_dino_chexpert_pre_adapted_7cohort",
    "projection_state_dict": model.projection.state_dict(),
    "feature_fusion_state_dict": model.fusion.state_dict(),
    "encoder_trainable_state_dict": encoder_state,
    "classification_head_state_dict": model.classifier.state_dict(),
    "cohorts": COHORTS,
    "training": {
        "dataset": "CheXpert frontal", "epochs": EPOCHS, "lr_head": LR_HEAD,
        "lr_encoder": LR_ENCODER, "batch_size": BATCH_SIZE, "steps_per_epoch": STEPS_PER_EPOCH,
        "uncertain_policy": UNCERTAIN_POLICY, "normal_definition": "No Finding=1 without target positive/uncertain; fallback=no target positive/uncertain",
        "balanced_sampling": True, "hard_negative_mining": True,
        "fine_tuned_blocks": UNFREEZE_LAST_BLOCKS, "checkpoint_metric": "dev_nDCG@5",
        "best_dev_nDCG@5": best_score,
    },
}, CHECKPOINT_PATH)

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "history.csv", index=False)
valid_data = collect(valid_loader)
valid_table, valid_auroc, valid_auprc, valid_ndcg5 = dev_summary(valid_data)
valid_table.to_csv(OUT_DIR / "internal_valid_classification.csv", index=False)
(OUT_DIR / "internal_valid_summary.json").write_text(json.dumps({"macro_auroc": valid_auroc, "macro_auprc": valid_auprc, "nDCG@5": valid_ndcg5}, indent=2))
print("Saved:", CHECKPOINT_PATH)
print("Internal validation nDCG@5:", valid_ndcg5)


## Use after CheXpert pre-adaptation

Keep this CheXpert checkpoint fixed during research evaluation. For
IU-Xray 5-fold evaluation, initialize each fold from this checkpoint,
train only the projection/fusion head (or a small LoRA adapter) on that
fold's IU-Xray training studies, then evaluate only held-out studies.
Do not use the IU-Xray test fold for checkpoint selection, thresholds,
gallery construction or fine-tuning.
